# Experiment 2 — Is the deviation causally necessary, *specifically* for modular computation?

Upgrades over the current paper result, each targeting a named weakness:
1. **Selectivity control** (the real confound): identical intervention, evaluated on
   *non-modular* tasks over the same tokens. If those are equally impaired, the
   deviation is generic identity code and the claim must be reframed.
2. **Well-defined null**: N ≥ 100 independent energy-matched random-deviation draws;
   gap, z, and empirical p computed against that distribution.
3. **Stronger control**: per-mode spectrum-preserving rotation of the deviation
   (preserves per-mode energy AND singular spectrum; changes only orientation).
4. **Third task family**: units-digit arithmetic mod 10 — Feucht et al.'s home turf.
5. **Surface area**: ≥5 templates per task, all operand×offset pairs, per-template
   reporting, cluster bootstrap over items, Page trend test for dose-response.

**GPU required for the model sections** (A100 recommended for 7B+ in bf16).
The synthetic self-test cell runs on CPU and must PASS before any model work.

---
## PRE-REGISTERED ANALYSIS PLAN — frozen before running.

**Interventions** (all preserve mode-0 and mode-1 exactly; asserted at runtime):
intact / circle_only / matched_random ×N / rotated_within_modes ×N / dose(α),
dose_ctrl(α) for α ∈ {0, .25, .5, .75, 1} / zero_rows (positive control).
Input embeddings only; if embeddings are tied, the output head is untied first.

**Primary outcomes.**
- P1 (necessity, not energy): acc(intact) − mean acc(matched_random) > 0;
  empirical p = (1 + #{draws ≥ intact}) / (N+1); report per model × task.
- P2 (orientation, not spectrum): acc(intact) − mean acc(rotated_within_modes) > 0.
  If P2 holds, the computation reads the deviation's *orientation* — strictly
  stronger than P1.
- P3 (selectivity): DiD = [modular drop] − [non-modular drop] > 0 under the same
  intervention, cluster-bootstrap 95% CI excluding 0. **If DiD ≈ 0, the honest
  conclusion is generic-identity damage and the paper's claim must be narrowed.**
- P4 (dose): Page's L one-sided across α on the true-deviation curve; matched
  control curve flat (Page |z| small).

**Decision rule (pre-committed).** The claim 'the deviation is causally necessary for
modular arithmetic' requires P1 AND P3. P2 upgrades it. Any model×task failing the
positive control (zero_rows not at chance) is excluded and reported as excluded.


In [ ]:
# ---------------- configuration ----------------
SMOKE = True    # True: 1 model, months only, N=25 draws. False: full grid.

MODELS = ([
    "Qwen/Qwen2.5-7B",
] if SMOKE else [
    "meta-llama/Llama-3.1-8B",
    "Qwen/Qwen2.5-7B",
    "mistralai/Mistral-7B-v0.3",
    "Qwen/Qwen2.5-14B",
])

N_DRAWS   = 25 if SMOKE else 100     # matched-random and rotation draws
ALPHAS    = [0.0, 0.25, 0.5, 0.75, 1.0]
TASKS     = (["months"] if SMOKE else ["months", "days", "digits"])
SEED      = 20260706
BATCH     = 16
import numpy as np
rng = np.random.default_rng(SEED)
print("SMOKE:", SMOKE, "| models:", MODELS, "| N_DRAWS:", N_DRAWS)

In [ ]:
"""Experiment 2 core (LLM deviation causality). Model-agnostic pieces:
Fourier decomposition over a cyclic vocabulary, the intervention family,
and the statistics. All validated on synthetic ground truth before use.

Decomposition convention (frozen):
  X in R^{n x d}: rows = token embeddings for the n cyclic items.
  Real Fourier basis F over Z/n: mode 0 (constant), modes 1..floor(n/2)
  (cos/sin pairs; the Nyquist mode is 1-dim for even n).
  circle    = mode k=1
  deviation = modes k>=2
  mode 0 (mean) is ALWAYS preserved by every intervention.

Interventions (all preserve mode 0 and mode 1 exactly; asserted):
  intact                  X unchanged
  circle_only             deviation zeroed
  matched_random(draw)    deviation replaced by random content with the SAME
                          per-mode Frobenius energy (N independent draws)
  rotated_within_modes    deviation rotated by an independent Haar rotation
                          inside each mode's token-subspace: preserves the
                          per-mode energy AND each mode's singular spectrum,
                          changes only orientation (strongest control)
  dose(alpha)             circle + alpha * true deviation
  dose_ctrl(alpha, draw)  circle + alpha * matched random deviation
  zero_rows               positive control: token rows zeroed
"""
import numpy as np


def fourier_modes(n):
    """Returns list of (k, B_k) with B_k an orthonormal basis (n x dim_k) of
    mode k's token subspace. Sum of dims = n."""
    j = np.arange(n)
    modes = [(0, np.ones((n, 1)) / np.sqrt(n))]
    for k in range(1, n // 2 + 1):
        c = np.cos(2 * np.pi * k * j / n)
        s = np.sin(2 * np.pi * k * j / n)
        M = np.column_stack([c, s]) if (2 * k != n) else c[:, None]
        Q, _ = np.linalg.qr(M)
        modes.append((k, Q))
    total = sum(B.shape[1] for _, B in modes)
    assert total == n
    return modes


class CyclicDecomposition:
    def __init__(self, n):
        self.n = n
        self.modes = fourier_modes(n)
        self.P = {k: B @ B.T for k, B in self.modes}

    def components(self, X):
        return {k: self.P[k] @ X for k, _ in self.modes}

    def mode_energies(self, X):
        return {k: float(np.linalg.norm(self.P[k] @ X) ** 2) for k, _ in self.modes}

    # ------------------------- interventions -------------------------
    def circle_only(self, X):
        return self.P[0] @ X + self.P[1] @ X

    def matched_random(self, X, rng):
        out = self.P[0] @ X + self.P[1] @ X
        for k, B in self.modes:
            if k < 2:
                continue
            target = np.linalg.norm(self.P[k] @ X)
            R = B @ rng.standard_normal((B.shape[1], X.shape[1]))
            nr = np.linalg.norm(R)
            if nr > 0:
                out = out + R * (target / nr)
        return out

    def rotated_within_modes(self, X, rng):
        out = self.P[0] @ X + self.P[1] @ X
        for k, B in self.modes:
            if k < 2:
                continue
            dk = B.shape[1]
            coords = B.T @ X                       # dk x d
            A = rng.standard_normal((dk, dk))
            Q, Rr = np.linalg.qr(A)
            Q = Q * np.sign(np.diag(Rr))
            out = out + B @ (Q @ coords)
        return out

    def dose(self, X, alpha):
        dev = X - self.P[0] @ X - self.P[1] @ X
        return self.P[0] @ X + self.P[1] @ X + alpha * dev

    def dose_ctrl(self, X, alpha, rng):
        rand_full = self.matched_random(X, rng)
        dev = rand_full - self.P[0] @ rand_full - self.P[1] @ rand_full
        return self.P[0] @ X + self.P[1] @ X + alpha * dev

    def check_circle_preserved(self, X, Y, tol=1e-8):
        for k in (0, 1):
            if not np.allclose(self.P[k] @ X, self.P[k] @ Y, atol=tol):
                return False
        return True


# ----------------------------- statistics -----------------------------------
def bootstrap_ci_accuracy(correct, cluster_ids, n_boot=5000, seed=0, alpha=0.05):
    """Cluster bootstrap over items (a 'cluster' = one base item, so prompts
    sharing an operand are resampled together)."""
    rng = np.random.default_rng(seed)
    correct = np.asarray(correct, dtype=float)
    cluster_ids = np.asarray(cluster_ids)
    clusters = np.unique(cluster_ids)
    members = {c: np.where(cluster_ids == c)[0] for c in clusters}
    stats = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.choice(clusters, size=len(clusters), replace=True)
        idxs = np.concatenate([members[c] for c in pick])
        stats[b] = correct[idxs].mean()
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(correct.mean()), (float(lo), float(hi))


def draw_null_pvalue(acc_intact, acc_draws):
    """Empirical one-sided p for 'intact beats the matched-random null':
    fraction of random-deviation draws achieving >= intact accuracy."""
    acc_draws = np.asarray(acc_draws, dtype=float)
    return float((1 + np.sum(acc_draws >= acc_intact)) / (1 + len(acc_draws)))


def gap_and_z(acc_intact, acc_draws):
    mu, sd = float(np.mean(acc_draws)), float(np.std(acc_draws, ddof=1))
    z = (acc_intact - mu) / sd if sd > 0 else np.inf
    return acc_intact - mu, z


def page_trend_L(matrix):
    """Page's L for ordered alternatives; matrix subjects x conditions,
    conditions in hypothesized increasing order."""
    from scipy.stats import rankdata
    n, k = matrix.shape
    ranks = np.vstack([rankdata(row) for row in matrix])
    R = ranks.sum(axis=0)
    L = float(np.sum(np.arange(1, k + 1) * R))
    EL = n * k * (k + 1) ** 2 / 4
    VL = n * k ** 2 * (k + 1) * (k ** 2 - 1) / 144
    return L, float((L - EL) / np.sqrt(VL))


def selectivity_did(drop_modular, drop_control):
    """Difference-in-differences: (modular impairment) - (control-task
    impairment). Positive = damage selective to modular computation."""
    return float(drop_modular - drop_control)


## Synthetic self-test — run this FIRST, no GPU needed

Validates the complete statistical pipeline on a toy 'model' with known ground truth:
(a) detects a planted orientation-dependent effect, (b) matched-control dose curve is
flat, (c) **calibration**: when the toy model ignores the deviation, p is large and the
gap ≈ 0 — i.e., the pipeline does not manufacture effects. All three must pass.

In [ ]:
def synthetic_selftest(seed=0):
    rng = np.random.default_rng(seed)
    n, d = 12, 64
    dec = CyclicDecomposition(n)
    X = rng.standard_normal((n, d))
    true_dev = X - dec.P[0] @ X - dec.P[1] @ X
    def toy_acc(Y, uses=True, noise=0.02):
        dev = Y - dec.P[0] @ Y - dec.P[1] @ Y
        a = np.sum(true_dev * dev) / (np.linalg.norm(true_dev) * np.linalg.norm(dev) + 1e-12)
        return (0.30 + (0.28 * max(a, 0) if uses else 0.0)) + rng.normal(0, noise)

    for nm, Y in [("circle", dec.circle_only(X)),
                  ("matched", dec.matched_random(X, rng)),
                  ("rot", dec.rotated_within_modes(X, rng)),
                  ("dose", dec.dose(X, 0.5))]:
        assert dec.check_circle_preserved(X, Y), nm
    e0 = dec.mode_energies(X); e1 = dec.mode_energies(dec.matched_random(X, rng))
    assert all(abs(e0[k] - e1[k]) < 1e-6 * max(1, e0[k]) for k in e0)

    ai = toy_acc(X)
    draws = [toy_acc(dec.matched_random(X, rng)) for _ in range(100)]
    gap, z = gap_and_z(ai, draws); p = draw_null_pvalue(ai, draws)
    assert gap > 0.15 and p <= 0.02, (gap, p)

    a0 = toy_acc(X, uses=False)
    d0 = [toy_acc(dec.matched_random(X, rng), uses=False) for _ in range(100)]
    g0, _ = gap_and_z(a0, d0); p0 = draw_null_pvalue(a0, d0)
    assert abs(g0) < 0.05 and p0 > 0.05, (g0, p0)

    subj = np.array([[toy_acc(dec.dose(X, a)) for a in [0, .25, .5, .75, 1]]
                     for _ in range(15)])
    L, zL = page_trend_L(subj)
    assert zL > 2.5, zL
    print(f"SELF-TEST PASS  (effect gap={gap:.3f} z={z:.1f} p={p:.3f} | "
          f"null gap={g0:+.3f} p={p0:.2f} | dose Page z={zL:.1f})")

synthetic_selftest()

## Model utilities — UNTESTED AGAINST LIVE MODELS in this build
(standard HF APIs; verified for syntax/logic only — run the self-test first, then a
single model in SMOKE before committing GPU-hours)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from contextlib import contextmanager

def load_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, torch_dtype=torch.bfloat16, device_map="auto")
    model.eval()
    # PRE-REGISTERED: intervene on INPUT embeddings only. If tied, untie by
    # cloning the LM head so the output side is unaffected.
    if getattr(model.config, "tie_word_embeddings", False):
        head = model.get_output_embeddings()
        new_head = torch.nn.Linear(head.in_features, head.out_features, bias=False)
        new_head.weight = torch.nn.Parameter(head.weight.detach().clone())
        model.set_output_embeddings(new_head.to(head.weight.device, head.weight.dtype))
        model.config.tie_word_embeddings = False
        print("  [untied embeddings: intervention affects input side only]")
    return tok, model

@contextmanager
def swapped_rows(model, token_ids, new_rows):
    '''Temporarily replace input-embedding rows for token_ids with new_rows
    (numpy n x d), restoring afterwards even on exception.'''
    emb = model.get_input_embeddings().weight
    idx = torch.tensor(token_ids, device=emb.device)
    backup = emb.data[idx].clone()
    try:
        emb.data[idx] = torch.tensor(new_rows, dtype=emb.dtype, device=emb.device)
        yield
    finally:
        emb.data[idx] = backup

@torch.no_grad()
def score_prompts(model, tok, prompts, candidate_ids, batch=16):
    '''For each prompt, argmax over the restricted candidate set at the final
    position. Returns predicted candidate index per prompt.'''
    preds = []
    for i in range(0, len(prompts), batch):
        chunk = prompts[i:i + batch]
        enc = tok(chunk, return_tensors="pt", padding=True).to(model.device)
        out = model(**enc)
        last = enc.attention_mask.sum(dim=1) - 1
        logits = out.logits[torch.arange(len(chunk)), last]      # B x V
        sub = logits[:, candidate_ids]                            # B x C
        preds.extend(sub.argmax(dim=-1).tolist())
    return preds

## Cyclic vocabularies and single-token verification
Tokenizer-dependent: each item must map to exactly ONE token in the chosen surface
form (leading space matters). The cell tries variants and asserts a working one; if a
model fails, it is excluded for that task and reported.

In [ ]:
MONTHS = ["January","February","March","April","May","June","July",
          "August","September","October","November","December"]
DAYS   = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
DIGITS = [str(i) for i in range(10)]

def resolve_single_tokens(tok, words):
    for prefix in (" ", ""):
        ids = []
        ok = True
        for w in words:
            t = tok.encode(prefix + w, add_special_tokens=False)
            if len(t) != 1:
                ok = False; break
            ids.append(t[0])
        if ok:
            return prefix, ids
    return None, None

VOCABS = {"months": (MONTHS, 12), "days": (DAYS, 7), "digits": (DIGITS, 10)}

## Tasks — modular targets + selectivity controls (≥5 templates each)
The selectivity controls use the SAME tokens under NON-modular computations.
Deliberately excluded as a control: 'the month after X' (that is +1 mod 12).

In [ ]:
MOD_TEMPLATES = {
 "months": ["If it is {a} and {k} months pass, the month will be",
            "Starting from {a}, after {k} months it will be",
            "{k} months after {a} is",
            "The month {k} months later than {a} is",
            "Count forward {k} months from {a}: the result is"],
 "days":   ["If today is {a} and {k} days pass, the day will be",
            "Starting from {a}, after {k} days it will be",
            "{k} days after {a} is",
            "The day {k} days later than {a} is",
            "Count forward {k} days from {a}: the result is"],
 "digits": ["The last digit of {x} + {y} is",
            "The units digit of {x} plus {y} is",
            "{x} + {y}: the final digit of the sum is",
            "Adding {x} and {y}, the ones digit is",
            "The sum {x} + {y} ends in the digit"],
}

# selectivity controls: months only (extend to days later if needed)
SEASON = {m: s for m, s in zip(MONTHS,
    ["winter","winter","spring","spring","spring","summer",
     "summer","summer","autumn","autumn","autumn","winter"])}
CTRL_TASKS = {
 "season":  (["The season of {a} in the Northern Hemisphere is",
              "In the Northern Hemisphere, {a} falls in",
              "{a} belongs to the season of",
              "The Northern Hemisphere season during {a} is",
              "Seasonally, {a} in the north is part of"],
             ["winter","spring","summer","autumn"],
             lambda a: SEASON[a]),
 "monthnum":(["As a number from 1 to 12, {a} is month",
              "{a} is month number",
              "In the calendar, {a} is the month numbered",
              "Numerically, {a} is month",
              "The position of {a} in the year, as a number, is"],
             [str(i) for i in range(1, 13)],
             lambda a: str(MONTHS.index(a) + 1)),
 "ordinal": (["In the calendar year, does {a} come before {b}? Answer yes or no:",
              "Within a single year, is {a} earlier than {b}? Answer yes or no:",
              "Does {a} precede {b} in the calendar year? Answer yes or no:",
              "Reading the calendar January to December, is {a} before {b}? Answer yes or no:",
              "Yes or no: {a} occurs earlier in the year than {b}. Answer:"],
             ["yes", "no"],
             lambda a, b: "yes" if MONTHS.index(a) < MONTHS.index(b) else "no"),
}

def build_modular_grid(task, items):
    n = len(items); rows = []
    for ti, tpl in enumerate(MOD_TEMPLATES[task]):
        if task == "digits":
            # operands MUST be the single-digit tokens themselves, so the
            # corrupted embeddings actually appear in the input (two-digit
            # numbers are their own tokens and would bypass the intervention)
            for x in range(2, 10):
                for y in range(2, 10):
                    rows.append(dict(prompt=tpl.format(x=str(x), y=str(y)),
                                     answer=(x + y) % 10, item=x, template=ti))
        else:
            for a in range(n):
                for k in range(1, n):
                    rows.append(dict(prompt=tpl.format(a=items[a], k=k),
                                     answer=(a + k) % n, item=a, template=ti))
    return rows

## Main experiment loop
For each model × task: verify tokens → positive control → intact / circle / N matched
draws / N rotation draws / dose curves → selectivity controls under the SAME strongest
intervention. Results accumulate in `records` and are saved per model.

In [ ]:
import pandas as pd, itertools, json as _json

def eval_grid(model, tok, grid, prefix, answer_ids):
    prompts = [r["prompt"] for r in grid]
    preds = score_prompts(model, tok, prompts, answer_ids, batch=BATCH)
    return np.array([int(p == r["answer"]) for p, r in zip(preds, grid)]),            np.array([r["item"] for r in grid]),            np.array([r["template"] for r in grid])

all_records = []
for model_name in MODELS:
    print("="*70, "\n", model_name)
    tok, model = load_model(model_name)
    tok.pad_token = tok.pad_token or tok.eos_token
    emb = model.get_input_embeddings().weight

    for task in TASKS:
        items, n = VOCABS[task]
        prefix, token_ids = resolve_single_tokens(tok, items)
        if token_ids is None:
            print(f"  [{task}] EXCLUDED: no single-token form"); continue
        X = emb.data[torch.tensor(token_ids, device=emb.device)].float().cpu().numpy()
        dec = CyclicDecomposition(n)
        grid = build_modular_grid(task, items)
        answer_ids = token_ids   # answers are the same cyclic vocabulary

        def run(rows_np, tag):
            assert dec.check_circle_preserved(X, rows_np, tol=1e-4) or tag in ("intact","zero")
            with swapped_rows(model, token_ids, rows_np):
                corr, item, tmpl = eval_grid(model, tok, grid, prefix, answer_ids)
            acc, ci = bootstrap_ci_accuracy(corr, item, n_boot=1000, seed=SEED)
            all_records.append(dict(model=model_name, task=task, cond=tag,
                                    acc=acc, ci_lo=ci[0], ci_hi=ci[1]))
            return acc, corr

        acc_intact, _ = run(X, "intact")
        acc_zero, _   = run(np.zeros_like(X), "zero")           # positive control
        chance = 1.0 / n
        if acc_zero > chance + 2.5 * np.sqrt(chance*(1-chance)/len(grid)):
            print(f"  [{task}] positive control FAILED (zero-rows acc {acc_zero:.3f}); "
                  f"EXCLUDE and report"); continue
        acc_circle, _ = run(dec.circle_only(X), "circle")

        draws_m = [run(dec.matched_random(X, rng), f"matched_{i}")[0]
                   for i in range(N_DRAWS)]
        draws_r = [run(dec.rotated_within_modes(X, rng), f"rotated_{i}")[0]
                   for i in range(N_DRAWS)]
        gap_m, z_m = gap_and_z(acc_intact, draws_m); p_m = draw_null_pvalue(acc_intact, draws_m)
        gap_r, z_r = gap_and_z(acc_intact, draws_r); p_r = draw_null_pvalue(acc_intact, draws_r)
        print(f"  [{task}] intact={acc_intact:.3f} circle={acc_circle:.3f} "
              f"matched={np.mean(draws_m):.3f} (gap {gap_m:+.3f}, z {z_m:.1f}, p {p_m:.4f}) "
              f"rotated={np.mean(draws_r):.3f} (gap {gap_r:+.3f}, z {z_r:.1f}, p {p_r:.4f})")

        # dose-response
        dose_true, dose_ctrl = [], []
        for a in ALPHAS:
            dose_true.append(run(dec.dose(X, a), f"dose_{a}")[0])
            dose_ctrl.append(run(dec.dose_ctrl(X, a, rng), f"dosectrl_{a}")[0])
        print(f"  [{task}] dose true {np.round(dose_true,3)} ctrl {np.round(dose_ctrl,3)}")

        # selectivity (months only)
        if task == "months":
            drop_mod = acc_intact - float(np.mean(draws_m))
            for cname, (tpls, cands, ans_fn) in CTRL_TASKS.items():
                cand_prefix, cand_ids = resolve_single_tokens(tok, cands)
                if cand_ids is None: 
                    print(f"    ctrl {cname}: EXCLUDED (multi-token answers)"); continue
                cg = []
                for ti, tpl in enumerate(tpls):
                    if cname == "ordinal":
                        for a, b in itertools.permutations(range(12), 2):
                            cg.append(dict(prompt=tpl.format(a=MONTHS[a], b=MONTHS[b]),
                                           answer=cands.index(ans_fn(MONTHS[a], MONTHS[b])),
                                           item=a, template=ti))
                    else:
                        for a in range(12):
                            cg.append(dict(prompt=tpl.format(a=MONTHS[a]),
                                           answer=cands.index(ans_fn(MONTHS[a])),
                                           item=a, template=ti))
                def acc_ctrl(rows_np):
                    with swapped_rows(model, token_ids, rows_np):
                        preds = score_prompts(model, tok, [r["prompt"] for r in cg],
                                              cand_ids, batch=BATCH)
                    return float(np.mean([int(p == r["answer"]) for p, r in zip(preds, cg)]))
                ci_ = acc_ctrl(X)
                cm = float(np.mean([acc_ctrl(dec.matched_random(X, rng))
                                    for _ in range(max(10, N_DRAWS // 5))]))
                did = selectivity_did(drop_mod, ci_ - cm)
                all_records.append(dict(model=model_name, task=f"ctrl_{cname}",
                                        cond="did", acc=did, ci_lo=ci_, ci_hi=cm))
                print(f"    selectivity vs {cname}: ctrl intact={ci_:.3f} corrupted={cm:.3f} "
                      f"drop={ci_-cm:+.3f} | modular drop={drop_mod:+.3f} | DiD={did:+.3f}")

    pd.DataFrame(all_records).to_csv("exp2_records.csv", index=False)
    del model; torch.cuda.empty_cache()
print("saved exp2_records.csv")

In [ ]:
# Trend tests + report table
rec = pd.DataFrame(all_records)
for (m, t), g in rec[rec.cond.str.startswith("dose_")].groupby(["model", "task"]):
    curve = g.sort_values("cond").acc.values
    print(m, t, "dose curve:", np.round(curve, 3),
          "(fit Page's L on per-item curves in the full run; grid-level curve shown here)")
print(rec.to_string(index=False))

## Decision rules (pre-committed, repeated so they sit next to the numbers)
- **P1 AND P3 hold** → 'the deviation is causally necessary for modular arithmetic'
  stands, now with the confound closed.
- **P1 holds, P3 fails** (control tasks equally impaired) → reframe: the deviation is
  general-purpose identity structure that modular computation *also* uses; the
  paper's claim narrows accordingly. Better discovered here than in review.
- **P2 also holds** → upgrade the claim: the computation reads the deviation's
  orientation, not any spectral property.
- **Perplexity check** (add if P3 is borderline): neutral month-mentioning text
  perplexity under intact vs matched — large degradation implies generic damage.

### Runtime notes
- 7B bf16 on A100-40GB: months grid = 5 templ × 12 × 11 = 660 prompts/condition;
  conditions ≈ 2 + 2·N + 10 ≈ 220 at N=100 → ~145k forward passes/model/task.
  Batch 16, ~0.15 s/batch → ~20–25 min per model per task. Four models × three
  tasks ≈ 4–5 GPU-hours. SMOKE ≈ 15 minutes.
- If Colab disconnects: `exp2_records.csv` is written after every model.